### [Delta Lake Internals #0 — Setting Up Delta Lake on Google Colab](https://medium.com/@swarajgupta/delta-lake-internals-0-setting-up-delta-lake-on-google-colab-58acace23433)

## configurations

In [1]:
pip install -q pyspark==3.4.1 delta-spark==2.4.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 MB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 10.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.2 requires pyspark[connect]~=4.0.0, but you have pyspark 3.4.1 which is incompatible.


In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
builder = (
    SparkSession.builder
    .appName("DeltaLakeApp")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()


In [3]:
import importlib.metadata
print("spark version : ",spark.version)
print("delta version : ",importlib.metadata.version("delta-spark"))

spark version :  3.4.1
delta version :  2.4.0


## start from here

In [4]:
df = spark.read.format('csv').option("header", "true")\
    .option("inferSchema","true")\
    .load("/content/sample_data/california_housing_train.csv")

df.printSchema()

root
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- median_house_value: double (nullable = true)



In [5]:
spark.sql("create database deltahub")

DataFrame[]

In [6]:
spark.sql("show databases").show()

+---------+
|namespace|
+---------+
|  default|
| deltahub|
+---------+



In [7]:
df.write.mode("overwrite").format("delta").saveAsTable("deltahub.housing")

In [8]:
spark.sql("select * from deltahub.housing").show(5)

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+
|  -114.31|   34.19|              15.0|     5612.0|        1283.0|    1015.0|     472.0|       1.4936|           66900.0|
|  -114.47|    34.4|              19.0|     7650.0|        1901.0|    1129.0|     463.0|         1.82|           80100.0|
|  -114.56|   33.69|              17.0|      720.0|         174.0|     333.0|     117.0|       1.6509|           85700.0|
|  -114.57|   33.64|              14.0|     1501.0|         337.0|     515.0|     226.0|       3.1917|           73400.0|
|  -114.57|   33.57|              20.0|     1454.0|         326.0|     624.0|     262.0|        1.925|           65500.0|
+---------+--------+----